# T05: Distribution Overrides

## What You'll Learn

Spindle's domain models ship with **default distribution profiles** that control the
probability weights for categorical columns (e.g., loyalty tier, gender, region).
Sometimes the defaults don't match your scenario — maybe you need a dataset skewed
toward premium customers, or one that mirrors a specific market segment.

In this tutorial you will:

1. **Generate** retail data with the default profile and inspect the distributions
2. **Override** a specific distribution using the `overrides` parameter
3. **Compare** before and after distributions side by side
4. **Explore** the `available_profiles` and `profile_name` properties
5. **Create** a custom profile JSON file

## Prerequisites

- `sqllocks-spindle` installed
- Familiarity with `Spindle.generate()` (see **T01**)

## Time Estimate

**~10 minutes**

In [ ]:
# Uncomment the line below if sqllocks-spindle is not yet installed.
# %pip install sqllocks-spindle

print("Installation cell ready. Uncomment the pip line above if needed.")

## Step 1 — Generate Retail Data with the Default Profile

**What we're about to do:** Generate a retail dataset using the default distribution
profile and inspect the `loyalty_tier` column on the `customer` table.

**Why this matters:** The default profile encodes Spindle's best-guess industry
distributions. Understanding what "default" looks like is the baseline for any
customization.

**What to expect:** A breakdown of loyalty tier counts and percentages reflecting
the default profile weights.

In [ ]:
from sqllocks_spindle import Spindle, RetailDomain

# Generate with default profile
domain_default = RetailDomain()
result_default = Spindle.generate(domain=domain_default, scale="fabric_demo", seed=42)

customers_default = result_default.tables["customer"]

print("Loyalty Tier Distribution — DEFAULT Profile")
print("=" * 50)
tier = customers_default["loyalty_tier"].value_counts()
tier_pct = customers_default["loyalty_tier"].value_counts(normalize=True).mul(100).round(1)
for val in tier.index:
    print(f"  {val:<12} {tier[val]:>5} customers  ({tier_pct[val]}%)")
print(f"\nTotal customers: {len(customers_default)}")

## Step 2 — Apply a Distribution Override

**What we're about to do:** Create a new `RetailDomain` with the `overrides` parameter
to shift the loyalty tier distribution. We'll make 50% of customers Gold tier — useful
for testing a loyalty-heavy analytics scenario.

**Why this matters:** The `overrides` parameter lets you tweak any categorical
distribution without editing profile files. Pass a dict of
`"table.column": {value: probability}` pairs and Spindle applies them at generation
time.

**What to expect:** A loyalty tier breakdown where Gold dominates at ~50%.

In [ ]:
# Override: shift loyalty_tier to be Gold-heavy
domain_override = RetailDomain(
    overrides={
        "customer.loyalty_tier": {
            "Basic": 0.30,
            "Gold": 0.50,
            "Platinum": 0.20,
        }
    }
)

result_override = Spindle.generate(domain=domain_override, scale="fabric_demo", seed=42)

customers_override = result_override.tables["customer"]

print("Loyalty Tier Distribution — OVERRIDDEN Profile")
print("=" * 50)
tier_o = customers_override["loyalty_tier"].value_counts()
tier_o_pct = customers_override["loyalty_tier"].value_counts(normalize=True).mul(100).round(1)
for val in tier_o.index:
    print(f"  {val:<12} {tier_o[val]:>5} customers  ({tier_o_pct[val]}%)")
print(f"\nTotal customers: {len(customers_override)}")

## Step 3 — Compare Before and After

**What we're about to do:** Display a side-by-side comparison of the default and
overridden distributions.

**Why this matters:** Seeing both profiles in one table makes it obvious how the
override shifted the data. This is exactly the kind of comparison you would show
stakeholders when proposing a test scenario.

In [ ]:
import pandas as pd

default_pct = customers_default["loyalty_tier"].value_counts(normalize=True).mul(100).round(1)
override_pct = customers_override["loyalty_tier"].value_counts(normalize=True).mul(100).round(1)

comparison = pd.DataFrame({
    "Default (%)": default_pct,
    "Overridden (%)": override_pct,
})
comparison["Delta"] = (comparison["Overridden (%)"] - comparison["Default (%)"]).round(1)
comparison = comparison.sort_index()

print("Loyalty Tier — Side-by-Side Comparison")
print("=" * 55)
print(comparison.to_string())
print(f"\nNote: probabilities in the override must sum to 1.0")

## Step 4 — Explore Available Profiles

**What we're about to do:** Use the `available_profiles` and `profile_name` properties
to discover what named profiles ship with the Retail domain.

**Why this matters:** Named profiles are pre-built distribution sets stored as JSON
files in the domain's `profiles/` directory. They're reusable, shareable, and version-
controllable — better than one-off overrides for recurring scenarios.

In [ ]:
domain = RetailDomain()

print(f"Active profile:    {domain.profile_name}")
print(f"Available profiles: {domain.available_profiles}")
print()

# You can load a specific named profile at construction time:
# domain = RetailDomain(profile="premium_heavy")
# This loads profiles/premium_heavy.json from the domain's directory.

## Step 5 — Create a Custom Profile JSON

**What we're about to do:** Write a custom profile JSON file that you could drop into
the domain's `profiles/` directory. This demonstrates the profile file format.

**Why this matters:** For repeatable, team-shared scenarios, a named profile JSON is
better than inline overrides. You write it once, commit it to source control, and
anyone can reference it by name.

In [ ]:
import json

custom_profile = {
    "name": "premium_heavy",
    "description": "Skewed toward premium loyalty tiers for VIP analytics testing",
    "distributions": {
        "customer.loyalty_tier": {
            "Basic": 0.10,
            "Silver": 0.15,
            "Gold": 0.40,
            "Platinum": 0.35,
        },
        "customer.region": {
            "Northeast": 0.40,
            "Southeast": 0.20,
            "Midwest": 0.15,
            "West": 0.15,
            "Southwest": 0.10,
        },
    },
    "ratios": {
        "orders_per_customer": 5.0,
    },
}

print("Custom profile JSON:")
print(json.dumps(custom_profile, indent=2))
print()
print("To use this profile, save it as:")
print("  <domain_path>/profiles/premium_heavy.json")
print()
print("Then load it with:")
print("  domain = RetailDomain(profile='premium_heavy')")

## What's Next?

You now know how to customize Spindle's data distributions using both inline overrides
and named profile files. Here's where to go from here:

| Notebook | What You'll Learn |
|----------|-------------------|
| **T06: Star Schema Export** | Transform generated data into dimension + fact tables and export to Parquet |
| **T07: Domain Quickstarts** | Generate and compare all 13 domains in a single notebook |
| **T04: Healthcare Deep Dive** | Explore a domain with complex clinical distributions |

**Key takeaways:**
- Use `overrides={}` for quick, one-off distribution changes
- Use `profile="name"` for repeatable, team-shared scenarios
- Check `domain.available_profiles` to see what ships out of the box
- Probabilities in an override dict should sum to 1.0